In [5]:
import pandas as pd
from pathlib import Path
import numpy as np
import anndata

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from abc_atlas_access.abc_atlas_cache.anndata_utils import get_gene_data

In [39]:
merfish_genes = pd.read_csv('../data/Xenium_V1_FF_Mouse_Brain_MultiSection_Input_gene_groups.csv')

In [7]:
download_base = Path('../data/ABA_WMB_atlas')
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

'releases/20250131/manifest.json'

In [43]:
cell = abc_cache.get_metadata_dataframe(directory='WMB-10X', file_name='cell_metadata')


In [44]:
cell.set_index('cell_label', inplace=True)

In [45]:
cell.dataset_label.value_counts()

dataset_label
WMB-10Xv3       2341350
WMB-10Xv2       1699939
WMB-10XMulti       1687
Name: count, dtype: int64

In [47]:
gene = abc_cache.get_metadata_dataframe(directory='WMB-10X', file_name='gene').set_index('gene_identifier')
gene

,gene_symbol,name,mapped_ncbi_identifier,comment
gene_identifier,,,,
ENSMUSG00000051951,Xkr4,X-linked Kx blood group related 4,NCBIGene:497097,NaN
ENSMUSG00000089699,Gm1992,predicted gene 1992,NaN,NaN
ENSMUSG00000102331,Gm19938,"predicted gene, 19938",NaN,NaN
ENSMUSG00000102343,Gm37381,"predicted gene, 37381",NaN,NaN
ENSMUSG00000025900,Rp1,retinitis pigmentosa 1 (human),NCBIGene:19888,NaN
...,...,...,...,...
ENSMUSG00000095523,AC124606.1,PRAME family member 8-like,NCBIGene:100038995,no expression
ENSMUSG00000095475,AC133095.2,uncharacterized LOC545763,NCBIGene:545763,no expression
ENSMUSG00000094855,AC133095.1,uncharacterized LOC620639,NCBIGene:620639,no expression


In [46]:
# Only get cells whose dataset_label starts with WMB-10Xv (exclude the multiome data)
cell = cell[cell.dataset_label.str.startswith('WMB-10XMulti')]
cell.shape

(1687, 16)

In [ ]:
# Let's try only the multimodal cells
cell = cell[cell.dataset_label.str.startswith('WMB-10Xv')]

In [40]:
# Get gene_names that are not in the gene dataframe
print(merfish_genes.index[~merfish_genes.index.isin(gene.index)])

# Is Hs3st2 in the gene dataframe?
print('Hs3st2' in gene.index)

# Replace Hs3St2 with Hs3st2
merfish_genes.index = merfish_genes.index.str.replace('Hs3St2', 'Hs3st2')

# Are all genes in the gene dataframe?
print(merfish_genes.index[merfish_genes.index.isin(gene.index)].all())

Index(['Hs3St2'], dtype='object')
True
True


In [48]:
gene_data = get_gene_data(
    abc_atlas_cache=abc_cache,
    all_cells=cell,
    all_genes=gene,
    selected_genes=list(merfish_genes.index),
    data_type='raw'
)

loading file: WMB-10XMulti


WMB-10XMulti-log2.h5ad: 100%|██████████| 89.3M/89.3M [00:42<00:00, 2.08MMB/s]  


 - time taken:  0.16165107700000192
total time taken: 1.704313942000006
	total cells: 1687 processed cells: 1687
